In [1]:
import numpy as np
import os

filename = 'huge.raw'

print("--- Setting up dummy file ---")
# Create a 64 MB file on disk (4000 x 4000 float32s = 64,000,000 bytes)
dummy_file = np.memmap(filename, dtype='float32', mode='w+', shape=(4000, 4000))
dummy_file.flush() # Ensure it writes to disk
del dummy_file     # Close the file

print("\n--- Running Exam Question Code ---")
# The exact code from your exam question:
big = np.memmap(filename, mode='r', dtype='float32', shape=(4000, 4000))
small = big[::4, ::4]

print(f"RAM used by small: {small.nbytes} bytes")

print("\n--- Cleaning up ---")
del small
del big
os.remove(filename)
print("File deleted.")

--- Setting up dummy file ---

--- Running Exam Question Code ---
RAM used by small: 4000000 bytes

--- Cleaning up ---
File deleted.


In [2]:
import inspect, blosc
print(inspect.getsource(blosc.compress))

def compress(bytesobj, typesize=8, clevel=9, shuffle=blosc.SHUFFLE,
             cname='blosclz'):
    """compress(bytesobj[, typesize=8, clevel=9, shuffle=blosc.SHUFFLE, cname='blosclz']])

    Compress bytesobj, with a given type size.

    Parameters
    ----------
    bytesobj : bytes-like object (supporting the buffer interface)
        The data to be compressed.
    typesize : int
        The data type size.
    clevel : int (optional)
        The compression level from 0 (no compression) to 9
        (maximum compression).  The default is 9.
    shuffle : int (optional)
        The shuffle filter to be activated.  Allowed values are
        blosc.NOSHUFFLE, blosc.SHUFFLE and blosc.BITSHUFFLE.  The
        default is blosc.SHUFFLE.
    cname : string (optional)
        The name of the compressor used internally in Blosc. It can be
        any of the supported by Blosc ('blosclz', 'lz4', 'lz4hc',
        'snappy', 'zlib', 'zstd' and maybe others too). The default is
        'blosc

In [3]:
help(np.sum)

Help on _ArrayFunctionDispatcher in module numpy:

sum(a, axis=None, dtype=None, out=None, keepdims=<no value>, initial=<no value>, where=<no value>)
    Sum of array elements over a given axis.

    Parameters
    ----------
    a : array_like
        Elements to sum.
    axis : None or int or tuple of ints, optional
        Axis or axes along which a sum is performed.  The default,
        axis=None, will sum all of the elements of the input array.  If
        axis is negative it counts from the last to the first axis. If
        axis is a tuple of ints, a sum is performed on all of the axes
        specified in the tuple instead of a single axis or all the axes as
        before.
    dtype : dtype, optional
        The type of the returned array and of the accumulator in which the
        elements are summed.  The dtype of `a` is used by default unless `a`
        has an integer dtype of less precision than the default platform
        integer.  In that case, if `a` is signed then t

In [4]:
a = np.float16(1000)
print(np.square(a))

inf


/var/folders/9z/28_5jgd51rqcszwnn7_b0wk00000gn/T/ipykernel_19560/1629861868.py:2: RuntimeWarning: overflow encountered in square
  print(np.square(a))


In [5]:
a = np.float16(10000)
b = np.float16(5)
print(a+b)
print(int(a+b))

1.001e+04
10008


In [6]:
X = np.array([
    [1,2,3,4],
    [5,6,7,8],
    [9,10,11,12],
    [13,14,15,16]
])
k = 1
n = X.shape[0]
print(f"A: {X.reshape(-1)[k:-n*k:n+1]}")
print(f"B: {X[k:-k]}")
print(f"C: {X.reshape(-1)[::k]}")
print(f"D: {X[k::n+1]}")

A: [ 2  7 12]
B: [[ 5  6  7  8]
 [ 9 10 11 12]]
C: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16]
D: [[5 6 7 8]]


In [7]:
FLOP = 6000
total_time = (166+232+158)*10**(-3)
FLOPs = FLOP / total_time
print(FLOPs)

10791.36690647482


In [9]:
x = np.memmap('memmap.raw', mode='r+', shape=(10, 10), dtype='float64')
print(x[0, 1])
x += 2

16.0


In [9]:
def colsum(x):
    return x.sum(axis=0)

a4 = np.random.rand(4, 4)
a8 = np.random.rand(8, 8)
a65536 = np.random.rand(65536, 65536)

In [34]:
from time import perf_counter
start = perf_counter()
colsum(a4)
a4_time = perf_counter()-start
print(a4_time)

5.8040954172611237e-05


In [43]:
start = perf_counter()
colsum(a8)
a8_time = perf_counter()-start
print(a8_time)

7.045827805995941e-05


In [15]:
start = perf_counter()
colsum(a65536)
a65536_time = perf_counter()-start
print(a65536_time)

187.84867833415046


In [24]:
a16 = np.random.rand(16, 16)
start = perf_counter()
colsum(a16)
a16_time = perf_counter()-start
print(a16_time)

5.116686224937439e-05


In [44]:
print(4**2/a4_time)
print(8**2/a8_time)
print(16**2/a16_time)
print(65526**2/a65536_time)

275667.4184303846
908338.9739603987
5003238.204295595
22856996.993943837


In [72]:
compute_t = 6.032
preprocess_t = 0.980
print(compute_t-compute_t/8)
print(compute_t-compute_t/4+preprocess_t-preprocess_t/4)

5.2780000000000005
5.2589999999999995


In [73]:
compute_t = 8.005
preprocess_t = 1.005
print(compute_t-compute_t/8)
print(compute_t-compute_t/4+preprocess_t-preprocess_t/4)

7.0043750000000005
6.7575


In [11]:
from numba import jit

@jit(nopython=True)
def localmax_1d(x, y, hw):
    for j in range(hw, len(x) - hw):
        m = -float('inf')
        for o in range(-hw, hw):
            m = max(x[j + o], m)
        y[j] = m
def localmax_rows(x, w):
    y = np.zeros_like(x)
    hw = w // 2
    for i in range(x.shape[0]):
        localmax_1d(x[i, :], y[i, :], hw)
    return y
def localmax_cols(x, w):
    y = np.zeros_like(x)
    hw = w // 2
    for i in range(x.shape[1]):
        localmax_1d(x[:, i], y[:, i], hw)
    return y

In [71]:
from time import perf_counter
a2000 = np.random.rand(2000, 2000)
start = perf_counter()
localmax_rows(a2000, 5)
print(perf_counter() - start)
start = perf_counter()
localmax_cols(a2000, 5)
print(perf_counter() - start)

0.013382124714553356
0.03251225035637617


In [ ]:
import numpy as np
a = np.float16(10000)
b = np.float16(5)

print(a + b)

1.001e+04


In [84]:
import numpy as np
a = np.float16(10000)
b = np.float16(5)
print(a + b)

1.001e+04
